# GOV-01 Version 2 Model Gate: historical experiment record

> **Closed experiment.** This notebook records the V2 workflow that produced the documented results. The protected test was used once on 2026-08-09 and is deliberately blocked below. Do not re-run training or protected-test cells for V2; start a new experiment with a new unused test split instead.

This notebook runs the first Version 2 model comparison on the Rome Road Damage Dataset. It uses only training and validation images. The protected test split stays unloaded until a V2 candidate is selected and locked.

The first real model is deliberately **unweighted**. A later experiment will test class weights as one controlled change.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['No_pothole', 'Pothole']
DATA_DIR = Path('data/processed/road_damage_rome_clean_split')
REPORTS_DIR = Path('reports')
REPORTS_DIR.mkdir(exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

for split_name in ['train', 'validation', 'test']:
    if not (DATA_DIR / split_name).is_dir():
        raise FileNotFoundError(f'Missing clean-split folder: {DATA_DIR / split_name}')

print('Clean V2 split found:', DATA_DIR.resolve())
print('Classes:', CLASS_NAMES)
print('Protected test folder exists but will not be loaded in this notebook.')

In [ ]:
def load_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )

train_raw = load_split('train', shuffle=True)
validation_ds = load_split('validation', shuffle=False)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name='v2_training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(
    lambda images, labels: (data_augmentation(images, training=True), labels),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

print('Training augmentation is enabled only for train_ds.')
print('Validation receives no random augmentation.')

In [ ]:
# Naive baseline: always predict the validation majority class, No_pothole.
# It is not trained. A real model should beat it, especially on macro F1.
y_validation = np.concatenate([labels.numpy().ravel() for _, labels in validation_ds])
naive_predictions = np.zeros_like(y_validation, dtype=int)
naive_scores = np.zeros_like(y_validation, dtype=float)

def calculate_metrics(y_true, predictions, scores):
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'macro_f1': float(f1_score(y_true, predictions, average='macro', zero_division=0)),
        'pothole_precision': float(precision_score(y_true, predictions, pos_label=1, zero_division=0)),
        'pothole_recall': float(recall_score(y_true, predictions, pos_label=1, zero_division=0)),
        'no_pothole_recall': float(recall_score(y_true, predictions, pos_label=0, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, scores)),
    }

naive_metrics = calculate_metrics(y_validation, naive_predictions, naive_scores)
display(pd.DataFrame([naive_metrics], index=['v2_naive_no_pothole']))

In [ ]:
# First real model: a compact CNN trained from scratch.
# Class weights are deliberately absent in this first baseline.
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability'),
], name='v2_cnn_unweighted')

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True,
)
cnn_model.summary()

In [ ]:
start_time = time.time()
history = cnn_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=20,
    callbacks=[early_stopping],
    verbose=1,
)
training_seconds = time.time() - start_time
print(f'Training time: {training_seconds:.1f} seconds')

In [ ]:
# Validation evaluation only. Do not replace validation_ds with test data here.
cnn_scores = cnn_model.predict(validation_ds).ravel()
cnn_predictions = (cnn_scores >= 0.50).astype(int)
cnn_metrics = calculate_metrics(y_validation, cnn_predictions, cnn_scores)

comparison = pd.DataFrame([
    {
        'run_name': 'v2_naive_no_pothole',
        'hypothesis': 'Always predicting the validation majority class establishes a reference floor.',
        'changed_factor': 'naive majority rule',
        'class_weight': 'not applicable',
        'training_seconds': 0.0,
        **naive_metrics,
    },
    {
        'run_name': 'v2_cnn_unweighted',
        'hypothesis': 'A compact CNN can learn road-image patterns beyond the naive majority rule.',
        'changed_factor': 'simple CNN model',
        'class_weight': 'none',
        'training_seconds': round(training_seconds, 1),
        **cnn_metrics,
    },
])
comparison.to_csv(REPORTS_DIR / 'v2_experiment_record.csv', index=False)
display(comparison)

print(classification_report(y_validation, cnn_predictions, target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_validation, cnn_predictions, display_labels=CLASS_NAMES)
plt.title('V2 validation confusion matrix: v2_cnn_unweighted')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'v2_cnn_unweighted_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
history_frame = pd.DataFrame(history.history)
history_frame[['loss', 'val_loss']].plot(title='V2 CNN training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'v2_cnn_unweighted_learning_curve.png', dpi=150)
plt.show()

print('Saved V2 Colab evidence in reports/:')
for path in sorted(REPORTS_DIR.glob('v2_*')):
    print('-', path.name)

print('Next decision: inspect validation macro F1 and both recalls before adding class weights.')

## Historical experiment 3 — class-weighted compact CNN

The class weights are calculated from the **training split only**. They make pothole mistakes count more during training because pothole images are less common. The observed validation result is stored in `reports/v2_experiment_record.csv`.

In [ ]:
def make_v2_datasets():
    train_raw_local = load_split('train', shuffle=True)
    validation_local = load_split('validation', shuffle=False)
    train_local = train_raw_local.map(
        lambda images, labels: (data_augmentation(images, training=True), labels),
        num_parallel_calls=AUTOTUNE,
    ).prefetch(AUTOTUNE)
    return train_raw_local, train_local, validation_local.prefetch(AUTOTUNE)

train_raw_weighted, train_ds_weighted, validation_ds_weighted = make_v2_datasets()
y_train_weighted = np.concatenate([labels.numpy().ravel() for _, labels in train_raw_weighted])
class_counts = np.bincount(y_train_weighted.astype(int), minlength=2)
class_weight = {index: len(y_train_weighted) / (2 * count) for index, count in enumerate(class_counts)}
print('Training counts:', dict(zip(CLASS_NAMES, class_counts)))
print('Class weights:', {CLASS_NAMES[key]: round(value, 3) for key, value in class_weight.items()})

cnn_weighted = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability'),
], name='v2_cnn_class_weighted')
cnn_weighted.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')])
cnn_weighted.summary()

In [ ]:
# Historical training cell. Do not run again: V2 is closed.
weighted_history = cnn_weighted.fit(
    train_ds_weighted, validation_data=validation_ds_weighted, epochs=20,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1,
)
weighted_scores = cnn_weighted.predict(validation_ds_weighted).ravel()
weighted_predictions = (weighted_scores >= 0.50).astype(int)
weighted_metrics = calculate_metrics(y_validation, weighted_predictions, weighted_scores)
display(pd.DataFrame([weighted_metrics], index=['v2_cnn_class_weighted']))
ConfusionMatrixDisplay.from_predictions(y_validation, weighted_predictions, display_labels=CLASS_NAMES)
plt.title('V2 validation confusion matrix: class-weighted CNN')
plt.show()

## Historical experiment 4 — frozen MobileNetV2 transfer learning

MobileNetV2 is pre-trained on a large image collection. **Frozen** means its feature-extractor layers are kept unchanged, while only the new pothole output layer learns from this dataset.

In [ ]:
train_raw_mobile, train_ds_mobile, validation_ds_mobile = make_v2_datasets()
mobile_base = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights='imagenet'
)
mobile_base.trainable = False
mobile_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input, name='mobilenetv2_preprocessing'),
    mobile_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability'),
], name='v2_mobilenetv2_frozen')
mobile_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')])
mobile_model.summary()

In [ ]:
# Historical training cell. Do not run again: V2 is closed.
frozen_history = mobile_model.fit(
    train_ds_mobile, validation_data=validation_ds_mobile, epochs=20,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1,
)
frozen_scores = mobile_model.predict(validation_ds_mobile).ravel()
frozen_predictions = (frozen_scores >= 0.50).astype(int)
frozen_metrics = calculate_metrics(y_validation, frozen_predictions, frozen_scores)
display(pd.DataFrame([frozen_metrics], index=['v2_mobilenetv2_frozen']))
ConfusionMatrixDisplay.from_predictions(y_validation, frozen_predictions, display_labels=CLASS_NAMES)
plt.title('V2 validation confusion matrix: frozen MobileNetV2')
plt.show()

## Historical experiment 5 — cautious MobileNetV2 fine-tuning

Fine-tuning lets only the final 10 MobileNetV2 feature layers adapt slowly. Batch-normalization layers remain frozen for stability. The learning rate is reduced to `1e-5` so the pre-trained image knowledge is not overwritten too quickly.

In [ ]:
# Historical fine-tuning setup. Do not run again: V2 is closed.
mobile_base.trainable = True
for layer in mobile_base.layers:
    layer.trainable = False
fine_tune_at = len(mobile_base.layers) - 10
for layer in mobile_base.layers[fine_tune_at:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

mobile_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')])
trainable_parameters = sum(variable.shape.num_elements() for variable in mobile_model.trainable_variables)
print('Fine-tuning begins at MobileNetV2 layer:', fine_tune_at)
print('Actual trainable parameters:', trainable_parameters)
mobile_model.summary()

In [ ]:
# Historical training cell. Do not run again: V2 is closed.
finetuned_history = mobile_model.fit(
    train_ds_mobile, validation_data=validation_ds_mobile, epochs=12,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1,
)
finetuned_scores = mobile_model.predict(validation_ds_mobile).ravel()
finetuned_predictions = (finetuned_scores >= 0.50).astype(int)
finetuned_metrics = calculate_metrics(y_validation, finetuned_predictions, finetuned_scores)
display(pd.DataFrame([finetuned_metrics], index=['v2_mobilenetv2_finetuned']))
print(classification_report(y_validation, finetuned_predictions, target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_validation, finetuned_predictions, display_labels=CLASS_NAMES)
plt.title('V2 validation confusion matrix: fine-tuned MobileNetV2')
plt.show()

## Historical validation-only threshold selection and candidate lock

The threshold changes when a probability becomes a `Pothole` prediction. It was selected from validation probabilities only. Macro F1 selected `0.50`; this configuration was locked before the one protected-test evaluation.

In [ ]:
# Historical analysis cell. Do not use it to tune V2 further.
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    predictions = (finetuned_scores >= threshold).astype(int)
    threshold_rows.append({'threshold': round(float(threshold), 2), **calculate_metrics(y_validation, predictions, finetuned_scores)})
threshold_results = pd.DataFrame(threshold_rows).sort_values('macro_f1', ascending=False)
display(threshold_results)
selected_threshold = 0.50
print('Historical selected validation threshold:', selected_threshold)
print('The recorded selected Macro F1 is 0.539904; see docs/v2_model_gate_plan.md.')

## Protected test — deliberately locked

The V2 test was evaluated once after the candidate lock. It is now closed. The recorded result is Macro F1 `0.479668`, pothole recall `0.841121`, no-pothole recall `0.258278`, and 112 false pothole alerts. See `docs/v2_model_gate_plan.md` and `reports/v2_experiment_record.csv`.

In [ ]:
# Safety lock: never change this for V2. A new experiment needs a new unused test split.
RUN_CLOSED_V2_PROTECTED_TEST = False
if RUN_CLOSED_V2_PROTECTED_TEST:
    raise RuntimeError('Blocked: the V2 protected test is closed. Create a new experiment and new test split instead.')
print('Protected V2 test remains locked. Historical one-time result is documented; no test data was loaded.')